# 04 - Model Training and Baseline Comparison

Fixed reproducible baselines using the validated target-free preprocessing pipeline. No final model is selected here.

## 1. Imports
Question: can all experiments share the same pipeline?

In [ ]:
from pathlib import Path
import pandas as pd
from src.models.train import baseline_models, run_experiments, write_results
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent

## 2. Load data
Question: is only processed training data used for training and selection?

In [ ]:
train = pd.read_csv(ROOT / 'data/processed/train_clean.csv')
train.shape, train.Transported.value_counts().to_dict()

## 3. Feature engineering
Question: does every classifier receive the same target-free engineering step?

In [ ]:
from src.features.engineering import TitanicFeatureEngineer
TitanicFeatureEngineer()

## 4. Train/validation split
Question: is the fixed 80/20 holdout stratified with random_state=42?

In [ ]:
from src.features.preprocessing import split_features_target
X_train, X_valid, y_train, y_valid = split_features_target(train)
len(X_train), len(X_valid), y_train.mean(), y_valid.mean()

## 5. Define preprocessing
Question: are imputation, scaling, and one-hot encoding fit inside each model pipeline?

In [ ]:
from src.features.preprocessing import make_feature_pipeline
make_feature_pipeline()

## 6. Define models
Question: which fixed baseline configurations are compared?

In [ ]:
baseline_models()

## 7. Train baseline models
Question: can all four pipelines fit without missing-value or convergence failures?

In [ ]:
results = run_experiments(train)
results['split']

## 8. Validation metrics
Question: what are the held-out accuracy, precision, recall, F1, and ROC-AUC values?

In [ ]:
results['validation'].round(4)

## 9. Cross-validation
Question: how do fixed baselines vary across five stratified folds?

In [ ]:
results['cross_validation'].round(4)

## 10. Model comparison
Question: which chart assets communicate validation comparisons without selecting a final model?

In [ ]:
sorted(path.name for path in (ROOT / 'web/assets/generated').glob('*model*.png'))

## 11. Feature importance
Question: what internal split-importance values does the fitted Random Forest expose?

In [ ]:
rf = results['fitted']['Random Forest']
pd.Series(rf.named_steps['model'].feature_importances_, index=rf.named_steps['preprocessing'].get_feature_names_out()).sort_values(ascending=False).head(15)

## 12. Save experiment results
Question: can measured tables be recorded without persisting experimental model binaries?

In [ ]:
write_results(results, ROOT / 'MODEL_RESULTS.md')

## 13. Conclusions
`MODEL_RESULTS.md` records measured holdout and cross-validation metrics. Step 05 will assess selection trade-offs; this notebook makes no final-model claim.